# Fraud Detection

![](https://news.mit.edu/sites/default/files/styles/news_article__image_gallery/public/images/201809/MIT-Fraud-Detection-PRESS_0.jpg?itok=n9A9HHwh)
[Img Source](https://news.mit.edu/2018/machine-learning-financial-credit-card-fraud-0920)

**This dataset is for "Financial dataset for Fraud Detection in a Company." It contains 9 columns:**

**step:** The number of minutes elapsed between each transaction.

**type:** The type of transaction - CASH_IN, CASH_OUT, DEBIT, PAYMENT, TRANSFER.

**amount:** The amount of money involved in the transaction.

**nameOrig:** The name of the person or entity originating the transaction.

**oldbalanceOrg:** The original balance of the account before the transaction took place.

**newbalanceOrig:** The balance of the account after the transaction was performed.

**nameDest:** The name of the person or entity receiving the transaction.

**oldbalanceDest:** The original balance of the recipient's account before the transaction took place.

**newbalanceDest:** The balance of the recipient's account after the transaction was performed.

**isFraud:** A binary label indicating whether the transaction was a fraudulent one or not.

The data in this dataset is used to detect fraud in financial transactions. It can be used to train machine learning models to identify fraudulent transactions and to prevent fraudulent activities in a company.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load the dataset
df = pd.read_csv("/kaggle/input/financial-dataset-for-fraud-detection-in-a-comapny/Fraud.csv", nrows=10000)

In [ ]:
df

In [ ]:
df.info()

# Data cleaning including missing values, outliers and multi-collinearity.

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().values.any()

In [ ]:
# Use fillna() to replace null values with the mean of the column
df = df.fillna(0)

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().values.any()

No Null Values in our Dataset.

In [ ]:
# Handle outliers
cols = ['amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest']
for col in cols:
    mean = df[col].mean()
    std = df[col].std()
    df = df[(df[col] > mean - 3*std) & (df[col] < mean + 3*std)]

For each column, the above code calculates the mean and standard deviation using the mean() and std() functions respectively. Next, it removes the rows that are more than 3 standard deviations away from the mean by applying a boolean mask to the dataframe using the mean and standard deviation calculated for each column. By doing this, we are removing the data points that are considered as outliers, which could be considered as noise or errors.

The z-score method is used here to detect the outliers, it's a standard method that compares each data point to the mean of the dataset and standard deviation, by doing this it can be determined whether a data point is an outlier or not. It's important to check the data distribution and correlation with the target variable before making any assumptions, also check the data types and make sure all columns are in the correct format to avoid any errors.

In [ ]:
df

# Data Visualization

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_style('whitegrid')

sns.barplot(x = 'isFraud', y = 'type', data = df).set(title = 'Fraud vs Transaction Type')
plt.show()

This code creates a bar plot using the Seaborn library. It sets the style of the plot to 'whitegrid'. Then, it uses the barplot() function to create a bar plot with the x-axis representing the 'isFraud' column and the y-axis representing the 'type' column, using the data from the 'df' DataFrame. The set() function is then used to set the title of the plot to 'Fraud vs Transaction Type'. Finally, the plt.show() function is used to display the plot. This code helps to visualize the relationship between the 'isFraud' column and the 'type' column in the dataset and can help in identifying the types of transactions that are more likely to be fraudulent.

# Fraud Detection Model

In [ ]:
# Import libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
df.columns

In [ ]:
# Define the predictor variables (X) and target variable (y)
X = df[['step', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'isFlaggedFraud']]
y = df['isFraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

This code defines the predictor variables (X) as the columns 'step', 'amount', 'oldbalanceOrg', 'oldbalanceDest', and 'isFlaggedFraud' in the dataframe 'df'. The target variable (y) is defined as the 'isFraud' column in the dataframe 'df'. Then, the data is split into training and testing sets using the train_test_split() function from the scikit-learn library. The test size is set to 0.25, meaning that 25% of the data will be used for testing and 75% of the data will be used for training. The random state is set to 42, which will ensure that the same data is used for testing and training each time the code is run.

In [ ]:
#Checking size of each df
print('X_train.shape :', X_train.shape)
print('X_test.shape :', X_test.shape)
print('y_train.shape :', y_train.shape)
print('y_test.shape :', y_test.shape)

In [ ]:
# Initialize the random forest classifier
rf_clf = RandomForestClassifier(random_state=42)

The code above initializes a random forest classifier object using the RandomForestClassifier class from the scikit-learn library. The random_state parameter is set to 42, which will ensure that the results are reproducible as the randomness in the model will be fixed to 42. This classifier object is now ready to be trained on data using the fit() method.

In [ ]:
# Train the model on the training data
rf_clf.fit(X_train, y_train)

In [ ]:
# Make predictions on the test data
y_pred = rf_clf.predict(X_test)

In [ ]:
# Print the classification report
print(classification_report(y_test, y_pred))

# Print the confusion matrix
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Print the accuracy of the model on the test data
print("Accuracy: {:.2f}%".format(rf_clf.score(X_test, y_test)*100))

In [ ]:
X_new = [[8, 109266.15, 10709.00, 1359.00, 0.0],
         [8, 1734.31, 20852.00, 0.00, 0.0],
         [9, 163537.34, 127693.33, 0.00, 0.0]]

y_pred = rf_clf.predict(X_new)
y_pred


In [ ]:
X_test

# Giving Input

In [ ]:
pip install joblib

In [ ]:
import joblib

# Save the model
joblib.dump(rf_clf, 'fraud_detection_model.pkl')

In [ ]:
import json
import pandas as pd
import joblib

def predict_fraud(input_json):
    # Define the input features relevant to your fraud detection model
    input_features = ['step','amount', 'oldbalanceOrg', 'oldbalanceDest', 'isFlaggedFraud']

    try:
        # Extract the relevant values from the JSON input
        input_data = [input_json[feature] for feature in input_features]

        # Create a DataFrame from the input data
        input_df = pd.DataFrame([input_data], columns=input_features)

        # Fill in missing values with zeros
        input_df = input_df.fillna(0)

        # Load your fraud detection model (replace 'fraud_detection_model.pkl' with your actual model filename)
        model = joblib.load('fraud_detection_model.pkl')

        # Make a prediction
        predicted_fraud = model.predict(input_df)

        if predicted_fraud[0] == 1:
            fraud_status = "This transaction is detected as FRAUDULENT."
        else:
            fraud_status = "This transaction is NOT fraudulent."

        return fraud_status
    except Exception as e:
        return str(e)

# Example JSON input for a transaction (you can modify this)
input_json = {"step": 1, "amount": 181.00, "oldbalanceOrg": 	181.0	, "oldbalanceDest": 21182.0	, "isFlaggedFraud": 0}

fraud_status = predict_fraud(input_json)
print(fraud_status)
